<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/stage_06_model_training/seq2one/stage_06_04_lstm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Stage_06_04 - T2 SEQ2ONE - LSTM**

# **SETUP COMÚN DEL PIPELINE DE ENTRENAMIENTO**

## **1. Imports**

In [1]:
# Permite anotaciones modernas en Python < 3.11
from __future__ import annotations

# ================================
# Standard library
# ================================
import os
import sys
import json
import time
import random
import importlib
import warnings
import logging
from pathlib import Path
from dataclasses import dataclass
from typing import Any, Dict, List, Iterable, Tuple

# ================================
# Third-party
# ================================
import numpy as np
import joblib

# ================================
# Configuración global
# ================================

# ---- Warnings (controlado, no agresivo)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# ---- Logging limpio para notebooks
logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)
logger.propagate = False  # evita duplicación con el root logger

# limpiar handlers si se re-ejecuta la celda
if logger.handlers:
    logger.handlers.clear()

handler = logging.StreamHandler()
formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
handler.setFormatter(formatter)
logger.addHandler(handler)

logger.info("Environment initialized")

2026-04-08 19:46:54,765 | INFO | Environment initialized


## **2. Acceso a drive**

In [2]:
# ================================
# Entorno (Google Drive / local)
# ================================

from pathlib import Path
import os

# Detectar si estamos en Colab
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

    DEFAULT_DRIVE_DIR = "/content/drive/MyDrive/neural_profit/"
else:
    # fallback local (puedes ajustarlo si quieres)
    DEFAULT_DRIVE_DIR = "./neural_profit/"

# Ruta base del proyecto
DRIVE_DIR = Path(os.environ.get("DRIVE_DIR", DEFAULT_DRIVE_DIR))

# Validación básica
if not DRIVE_DIR.exists():
    logger.warning(f"DRIVE_DIR no existe: {DRIVE_DIR}")
else:
    logger.info(f"DRIVE_DIR: {DRIVE_DIR}")

2026-04-08 19:47:24,463 | INFO | DRIVE_DIR: /content/drive/MyDrive/neural_profit


Mounted at /content/drive


## **3. Rutas de ventanas `seq2one` y `scaler`**

In [3]:
# ================================
# Configuración de paths
# ================================

WINDOWS_SEQ2ONE_DIR = DRIVE_DIR / os.environ.get(
    "WINDOWS_SEQ2ONE_DIR",
    "data/07_windows/seq2one/"
)

SCALERS_DIR = DRIVE_DIR / os.environ.get(
    "SCALERS_DIR",
    "data/06_scaled/"
)

# Validación básica
for p in [WINDOWS_SEQ2ONE_DIR, SCALERS_DIR]:
    if not p.exists():
        logger.warning(f"Path no existe: {p}")
    else:
        logger.info(f"Path OK: {p}")

# ================================
# Configuración del experimento
# ================================

# Targets T2 (clasificación)
TARGETS = [
    "t2_dir_thr_90",
    "t2_dir_thr_120",
]

# Tamaños de ventana
WINDOW_SIZES = [30, 60, 90, 120, 180]

# Splits
SPLITS = ["train", "valid", "test"]

# Features finales (consistencia global)
FEATURES_T2 = [
    "regime_id",
    "roc_30",
    "roc_60",
    "stoch_k_30",
    "atr_norm_10",
]

logger.info("Configuración de experimento cargada")
logger.info(f"Targets: {TARGETS}")
logger.info(f"Window sizes: {WINDOW_SIZES}")

2026-04-08 19:47:25,074 | INFO | Path OK: /content/drive/MyDrive/neural_profit/data/07_windows/seq2one
2026-04-08 19:47:25,074 | INFO | Path OK: /content/drive/MyDrive/neural_profit/data/06_scaled
2026-04-08 19:47:25,074 | INFO | Configuración de experimento cargada
2026-04-08 19:47:25,075 | INFO | Targets: ['t2_dir_thr_90', 't2_dir_thr_120']
2026-04-08 19:47:25,075 | INFO | Window sizes: [30, 60, 90, 120, 180]


In [4]:
# ================================
# Construcción y validación de paths (windows + scaler compartido)
# ================================

def build_windows_paths(window_sizes, targets, splits, base_dir):
    """
    windows_paths[w][t][s] -> Path
    """
    windows_paths = {}

    for w in window_sizes:
        windows_paths[w] = {}
        for t in targets:
            windows_paths[w][t] = {}
            for s in splits:
                windows_paths[w][t][s] = (
                    base_dir
                    / f"L{w}"
                    / f"windows_{t}_{s}.npz"
                )

    return windows_paths


def build_shared_scaler_path(base_dir, scaler_name="scaler_t2.pkl"):
    """
    Retorna el path del scaler compartido para T2.
    """
    return base_dir / scaler_name


# ================================
# Construcción
# ================================

WINDOWS_PATHS = build_windows_paths(
    window_sizes=WINDOW_SIZES,
    targets=TARGETS,
    splits=SPLITS,
    base_dir=WINDOWS_SEQ2ONE_DIR,
)

SCALER_T2_PATH = build_shared_scaler_path(
    base_dir=SCALERS_DIR,
    scaler_name="scaler_t2.pkl",
)

logger.info("Paths construidos (windows + scaler compartido T2)")

# ================================
# Validación
# ================================

missing_windows = []
existing_windows = []

for w in WINDOW_SIZES:
    for t in TARGETS:
        for s in SPLITS:
            p = WINDOWS_PATHS[w][t][s]
            if p.exists():
                existing_windows.append(p)
            else:
                missing_windows.append(p)

scaler_exists = SCALER_T2_PATH.exists()

# -------------------------------
# Logs
# -------------------------------
logger.info(f"Windows OK      : {len(existing_windows)}")
logger.info(f"Windows missing : {len(missing_windows)}")

if scaler_exists:
    logger.info(f"Scaler T2 OK    : {SCALER_T2_PATH}")
else:
    logger.warning(f"Scaler T2 missing: {SCALER_T2_PATH}")

if missing_windows:
    logger.warning("Archivos de ventanas faltantes:")
    for p in missing_windows[:5]:
        logger.warning(f"Missing window: {p}")

if not missing_windows and scaler_exists:
    logger.info("Todos los archivos existen correctamente")

# ================================
# Ejemplo de uso
# ================================

# window
# WINDOWS_PATHS[60]["t2_dir_thr_90"]["train"]

# scaler compartido
# SCALER_T2_PATH

2026-04-08 19:47:25,079 | INFO | Paths construidos (windows + scaler compartido T2)
2026-04-08 19:47:25,770 | INFO | Windows OK      : 30
2026-04-08 19:47:25,770 | INFO | Windows missing : 0
2026-04-08 19:47:25,770 | INFO | Scaler T2 OK    : /content/drive/MyDrive/neural_profit/data/06_scaled/scaler_t2.pkl
2026-04-08 19:47:25,771 | INFO | Todos los archivos existen correctamente


## **4. Carga de ventanas X/y**

### **4.1. Cargar ventanas (*.npz)**

In [5]:
from pathlib import Path
from typing import Tuple
import numpy as np


def load_npz_windows(path: Path) -> Tuple[np.ndarray, np.ndarray]:
    """
    Carga ventanas X e y desde un archivo .npz estándar (T2 seq2one).

    Espera:
    - 'X': (n_samples, seq_len, n_features)
    - 'y' o 'Y': (n_samples,) o (n_samples, 1) o (n_samples, seq_len, 1)

    Retorna
    -------
    X : np.ndarray  -> (n_samples, seq_len, n_features)
    y : np.ndarray  -> (n_samples,)
    """

    # --------------------------------------------------
    # 1. Validación
    # --------------------------------------------------
    if not path.exists():
        raise FileNotFoundError(f"No se encontró el archivo: {path}")

    # --------------------------------------------------
    # 2. Carga
    # --------------------------------------------------
    with np.load(path) as data:

        if "X" not in data:
            raise KeyError(f"NPZ inválido (falta clave 'X'): {path}")

        X = data["X"]

        if "y" in data:
            y = data["y"]
        elif "Y" in data:
            y = data["Y"]
        else:
            raise KeyError(f"NPZ inválido (falta clave 'y'/'Y'): {path}")

        # copiar a memoria
        X = X.copy()
        y = y.copy()

    # --------------------------------------------------
    # 3. Ajuste de shape de y (CRÍTICO)
    # --------------------------------------------------
    # Caso (n_samples, 1)
    if y.ndim == 2 and y.shape[1] == 1:
        y = y.reshape(-1)

    # Caso (n_samples, seq_len, 1) -> tomar último valor
    elif y.ndim == 3:
        y = y[:, -1, 0]

    # Caso (n_samples, seq_len) -> tomar último valor
    elif y.ndim == 2 and y.shape[1] > 1:
        y = y[:, -1]

    # Validación final
    if y.ndim != 1:
        raise ValueError(f"y no es 1D después de procesamiento: shape={y.shape}")

    # --------------------------------------------------
    # 4. Logs útiles
    # --------------------------------------------------
    logger.info(f"Loaded: {path.name}")
    logger.info(f"X shape: {X.shape} | y shape: {y.shape}")

    return X, y

### **4.2. Cargar escalador (*.pkl)**

In [6]:
# --------------------------------------------------
# Función común: carga scaler compartido (T2)
# --------------------------------------------------
def load_scaler(path: Path) -> Any:
    """
    Carga un scaler serializado con joblib.

    Parámetros
    ----------
    path : Path
        Ruta al archivo scaler (.pkl)

    Retorna
    -------
    scaler : Any
        Objeto scaler (ej. StandardScaler, MinMaxScaler)
    """

    # --------------------------------------------------
    # 1. Validación de existencia
    # --------------------------------------------------
    if not path.exists():
        raise FileNotFoundError(f"No se encontró el scaler: {path}")

    # --------------------------------------------------
    # 2. Carga
    # --------------------------------------------------
    scaler = joblib.load(path)

    # --------------------------------------------------
    # 3. Validación básica (opcional pero útil)
    # --------------------------------------------------
    if not hasattr(scaler, "transform"):
        raise TypeError(f"El objeto cargado no es un scaler válido: {type(scaler)}")

    # --------------------------------------------------
    # 4. Logging
    # --------------------------------------------------
    logger.info(f"Scaler cargado: {path.name}")

    return scaler

### **4.3. Función de carga**

In [7]:
from typing import Any, Dict, Mapping
from pathlib import Path


# --------------------------------------------------
# Carga completa: ventanas + scaler compartido T2
# --------------------------------------------------
def load_windows_and_scaler(
    *,
    window_size: int,
    target: str,
    windows_paths: Mapping[int, Mapping[str, Mapping[str, Path]]],
    scaler_path: Path,
) -> Dict[str, Any]:
    """
    Carga X/y (train/valid/test) y el scaler compartido T2
    para un par (window_size, target).

    Parámetros
    ----------
    window_size : int
        Tamaño de ventana.
    target : str
        Target T2, por ejemplo:
        - 't2_dir_thr_90'
        - 't2_dir_thr_120'
    windows_paths : mapping
        Estructura:
        windows_paths[window_size][target][split] -> Path
    scaler_path : Path
        Ruta al scaler compartido T2.

    Retorna
    -------
    bundle : dict
        Diccionario con:
        - metadata
        - paths
        - scaler
        - train/valid/test con X e y
    """

    # --------------------------
    # 1) Validaciones
    # --------------------------
    if window_size not in windows_paths:
        raise KeyError(f"window_size={window_size} no existe en windows_paths")

    if target not in windows_paths[window_size]:
        raise KeyError(f"target='{target}' no existe en windows_paths[{window_size}]")

    for split in ["train", "valid", "test"]:
        if split not in windows_paths[window_size][target]:
            raise KeyError(
                f"split='{split}' no existe en windows_paths[{window_size}]['{target}']"
            )

    if not scaler_path.exists():
        raise FileNotFoundError(f"No se encontró el scaler compartido: {scaler_path}")

    # --------------------------
    # 2) Paths
    # --------------------------
    train_path = windows_paths[window_size][target]["train"]
    valid_path = windows_paths[window_size][target]["valid"]
    test_path  = windows_paths[window_size][target]["test"]

    # --------------------------
    # 3) Carga de ventanas
    # --------------------------
    X_train, y_train = load_npz_windows(train_path)
    X_valid, y_valid = load_npz_windows(valid_path)
    X_test, y_test   = load_npz_windows(test_path)

    # --------------------------
    # 4) Carga de scaler
    # --------------------------
    scaler = load_scaler(scaler_path)

    # --------------------------
    # 5) Inferir horizonte
    # --------------------------
    try:
        horizon = int(target.split("_")[-1])
    except Exception:
        horizon = None

    # --------------------------
    # 6) Logging
    # --------------------------
    logger.info(
        f"Bundle cargado | target={target} | window_size={window_size} | "
        f"train={X_train.shape} | valid={X_valid.shape} | test={X_test.shape}"
    )

    # --------------------------
    # 7) Retorno
    # --------------------------
    return {
        "window_size": window_size,
        "target": target,
        "horizon": horizon,
        "paths": {
            "train": str(train_path),
            "valid": str(valid_path),
            "test": str(test_path),
            "scaler": str(scaler_path),
        },
        "scaler": scaler,
        "train": {
            "X": X_train,
            "y": y_train,
        },
        "valid": {
            "X": X_valid,
            "y": y_valid,
        },
        "test": {
            "X": X_test,
            "y": y_test,
        },
    }


### **4.4. Creación de bundles T2**

In [8]:
# --------------------------------------------------
# Crea bundles T2 para un window_size dado
# --------------------------------------------------
def create_bundles(
    window_size: int,
    targets: list[str] = TARGETS,
    windows_paths=WINDOWS_PATHS,
    scaler_path=SCALER_T2_PATH,
):
    """
    Crea un bundle por target para un window_size dado.

    Retorna
    -------
    bundle_t2_90  : dict
    bundle_t2_120 : dict
    """

    if len(targets) != 2:
        raise ValueError(
            f"Se esperaban exactamente 2 targets T2. Recibido: {targets}"
        )

    bundle_t2_90 = load_windows_and_scaler(
        window_size=window_size,
        target=targets[0],
        windows_paths=windows_paths,
        scaler_path=scaler_path,
    )

    bundle_t2_120 = load_windows_and_scaler(
        window_size=window_size,
        target=targets[1],
        windows_paths=windows_paths,
        scaler_path=scaler_path,
    )

    # --------------------------------------------------
    # Verificación rápida
    # --------------------------------------------------
    print(f"\n{'=' * 70}")
    print(f"WINDOW_SIZE: {window_size}")
    print(f"{'=' * 70}")

    print(f"\nTARGET: {targets[0]}")
    print("Train :", bundle_t2_90["train"]["X"].shape, bundle_t2_90["train"]["y"].shape)
    print("Valid :", bundle_t2_90["valid"]["X"].shape, bundle_t2_90["valid"]["y"].shape)
    print("Test  :", bundle_t2_90["test"]["X"].shape,  bundle_t2_90["test"]["y"].shape)
    print("Scaler:", type(bundle_t2_90["scaler"]).__name__)

    print(f"\nTARGET: {targets[1]}")
    print("Train :", bundle_t2_120["train"]["X"].shape, bundle_t2_120["train"]["y"].shape)
    print("Valid :", bundle_t2_120["valid"]["X"].shape, bundle_t2_120["valid"]["y"].shape)
    print("Test  :", bundle_t2_120["test"]["X"].shape,  bundle_t2_120["test"]["y"].shape)
    print("Scaler:", type(bundle_t2_120["scaler"]).__name__)

    return bundle_t2_90, bundle_t2_120

In [9]:
#bundle_t2_90, bundle_t2_120 = create_bundles(window_size=60)

Como acceder a las ventanas X e y:

```python
X_train_90 = bundle_t2_90["train"]["X"]
y_train_90 = bundle_t2_90["train"]["y"]

X_valid_120 = bundle_t2_120["valid"]["X"]
y_valid_120 = bundle_t2_120["valid"]["y"]

scaler = bundle_t2_90["scaler"]
```




### **4.5. Preparación de inputs según el tipo de modelo**

In [10]:
# ============================================================
# Preparación de inputs según el tipo de modelo
# ============================================================

def flatten_seq2one_X(X: np.ndarray) -> np.ndarray:
    """
    Aplana ventanas seq2one para modelos tabulares.

    Convierte:
        (n_samples, seq_len, n_features)
    a:
        (n_samples, seq_len * n_features)
    """
    X = np.asarray(X)

    if X.ndim != 3:
        raise ValueError(
            f"Se esperaba X 3D con shape (n, seq_len, n_features). "
            f"Recibido X.shape={X.shape}"
        )

    n_samples = X.shape[0]
    return X.reshape(n_samples, -1)


def prepare_X_for_model(
    X: np.ndarray,
    *,
    input_mode: str,
) -> np.ndarray:
    """
    Prepara X según el tipo de modelo.

    Parámetros
    ----------
    X : np.ndarray
        Array de entrada.
    input_mode : str
        - '2d_flat' : aplana ventanas 3D a 2D
        - '3d'      : deja X como está

    Retorna
    -------
    np.ndarray
        X transformado para el modelo correspondiente.
    """
    X = np.asarray(X)

    if input_mode == "3d":
        if X.ndim != 3:
            raise ValueError(
                f"Se esperaba X 3D para input_mode='3d'. "
                f"Recibido X.shape={X.shape}"
            )
        return X

    if input_mode == "2d_flat":
        return flatten_seq2one_X(X)

    raise ValueError(
        f"input_mode no soportado: {input_mode}. "
        f"Use '2d_flat' o '3d'."
    )

**Cómo se usa después**

Para Logistic Regression:

```python
X_train_model = prepare_X_for_model(bundle["train"]["X"], input_mode="2d_flat")
X_valid_model = prepare_X_for_model(bundle["valid"]["X"], input_mode="2d_flat")
```

Para LSTM / GRU / Transformer:

```python
X_train_model = prepare_X_for_model(bundle["train"]["X"], input_mode="3d")
X_valid_model = prepare_X_for_model(bundle["valid"]["X"], input_mode="3d")
```


## **5. Sanity Check**

In [11]:
from __future__ import annotations

from typing import Any, Optional, Tuple
import numpy as np


# ============================================================
# 1) Utilidades internas (inferencias y normalización)
# ============================================================

def _as_numpy_array(a: Any, *, name: str) -> np.ndarray:
    """
    Convierte una entrada a np.ndarray y valida que no esté vacía.
    """
    arr = np.asarray(a)

    if arr.size == 0:
        raise ValueError(f"{name} está vacío. shape={arr.shape}")

    if not np.isfinite(arr).all():
        raise ValueError(
            f"{name} contiene NaN/inf. Limpie antes de continuar. shape={arr.shape}"
        )

    return arr


def _normalize_y_seq2one(y: np.ndarray, *, name: str = "y") -> np.ndarray:
    """
    Normaliza y para seq2one a shape (n_samples,).

    Acepta:
    - (n,)
    - (n, 1)

    Rechaza shapes incompatibles.
    """
    if y.ndim == 1:
        return y

    if y.ndim == 2 and y.shape[1] == 1:
        return y.squeeze(1)

    raise ValueError(
        f"{name} shape inválido para seq2one. "
        f"Se esperaba (n,) o (n,1). Recibido {y.shape}"
    )


def _infer_seq_len_and_n_features(X: np.ndarray) -> Tuple[int, Optional[int], str]:
    """
    Infiere (seq_len, n_features, mode) desde X.

    mode:
    - '3d': X = (n, seq_len, n_features)
    - '2d': X = (n, d_flat)  -> n_features=None
    """
    if X.ndim == 3:
        return int(X.shape[1]), int(X.shape[2]), "3d"

    if X.ndim == 2:
        return int(X.shape[1]), None, "2d"

    raise ValueError(
        f"X.ndim inválido. Se esperaba 2D o 3D. Recibido X.shape={X.shape}"
    )

In [12]:
# ============================================================
# 2) Sanity check principal (seq2one clasificación T2)
# ============================================================

def sanity_check_seq2one(
    X: Any,
    y: Any,
    split_name: str,
    *,
    expected_seq_len: Optional[int] = None,
    expected_n_features: Optional[int] = None,
    expected_flat_dim: Optional[int] = None,
    allow_seq_inputs_take_last: bool = False,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Sanity check para datasets seq2one de clasificación.

    Soporta:
      - X 3D: (n, seq_len, n_features)
      - X 2D: (n, d_flat)  [ventana aplanada]

    y soporta:
      - y 1D: (n,)
      - y 2D: (n,1)

    Parámetros
    ----------
    expected_seq_len:
        Para X 3D: valida seq_len.
        Para X 2D: si expected_flat_dim no está, puede usarse como d_flat esperado.
    expected_n_features:
        Solo aplica a X 3D.
    expected_flat_dim:
        Solo aplica a X 2D (d_flat esperado).
    allow_seq_inputs_take_last:
        Si y viniera como (n, seq_len), permite tomar y[:, -1].
        Por defecto False.
    """

    X = _as_numpy_array(X, name=f"X[{split_name}]")
    y = _as_numpy_array(y, name=f"y[{split_name}]")

    # --------------------------------------------------
    # Normalización de y
    # --------------------------------------------------
    if y.ndim == 2 and y.shape[1] != 1 and allow_seq_inputs_take_last:
        y = y[:, -1]

    y = _normalize_y_seq2one(y, name=f"y[{split_name}]")

    # --------------------------------------------------
    # Inferir modo y dimensiones de X
    # --------------------------------------------------
    seq_len, n_features, mode = _infer_seq_len_and_n_features(X)
    n_samples = int(X.shape[0])

    # --------------------------------------------------
    # Validación básica de n_samples
    # --------------------------------------------------
    if y.shape[0] != n_samples:
        raise ValueError(
            f"Mismatch n_samples en {split_name}: "
            f"X tiene n={n_samples}, y tiene n={y.shape[0]}"
        )

    # --------------------------------------------------
    # Validación de shapes según modo
    # --------------------------------------------------
    if mode == "3d":
        if expected_seq_len is not None and seq_len != int(expected_seq_len):
            raise ValueError(
                f"{split_name}: seq_len esperado={expected_seq_len}, "
                f"recibido={seq_len}. X.shape={X.shape}"
            )

        if expected_n_features is not None and n_features != int(expected_n_features):
            raise ValueError(
                f"{split_name}: n_features esperado={expected_n_features}, "
                f"recibido={n_features}. X.shape={X.shape}"
            )

    elif mode == "2d":
        d_flat = int(X.shape[1])

        if expected_flat_dim is not None and d_flat != int(expected_flat_dim):
            raise ValueError(
                f"{split_name}: d_flat esperado={expected_flat_dim}, "
                f"recibido={d_flat}. X.shape={X.shape}"
            )

        if expected_flat_dim is None and expected_seq_len is not None and d_flat != int(expected_seq_len):
            raise ValueError(
                f"{split_name}: d_flat esperado={expected_seq_len}, "
                f"recibido={d_flat}. X.shape={X.shape}"
            )

        if expected_n_features is not None:
            raise ValueError(
                f"{split_name}: expected_n_features no aplica para X 2D. "
                f"Use expected_flat_dim o pase X en 3D."
            )

    # --------------------------------------------------
    # Diagnóstico de clases
    # --------------------------------------------------
    classes, counts = np.unique(y, return_counts=True)
    class_distribution = {
        str(cls): int(cnt) for cls, cnt in zip(classes, counts)
    }

    if len(classes) < 2:
        raise ValueError(
            f"{split_name}: y contiene menos de 2 clases únicas. "
            f"classes={classes.tolist()}"
        )

    # --------------------------------------------------
    # Salida informativa
    # --------------------------------------------------
    info = {
        "split": split_name,
        "X_shape": tuple(X.shape),
        "y_shape": tuple(y.shape),
        "mode": mode,
        "seq_len": seq_len if mode == "3d" else None,
        "n_features": n_features if mode == "3d" else None,
        "flat_dim": int(X.shape[1]) if mode == "2d" else None,
        "n_classes": int(len(classes)),
        "classes": classes.tolist(),
        "class_distribution": class_distribution,
    }

    if verbose:
        print(
            f"[sanity_check_seq2one] {split_name} | "
            f"X={info['X_shape']} | y={info['y_shape']} | "
            f"mode={mode} | n_classes={info['n_classes']} | "
            f"classes={info['classes']}"
        )

    return info

In [13]:
# ============================================================
# 3) Sanity checks por bundle (train/valid/test)
# ============================================================

def run_sanity_checks_for_bundle_seq2one(
    bundle: Dict[str, Any],
    *,
    tag: str,
    expected_seq_len: Optional[int] = None,
    expected_n_features: Optional[int] = None,
    expected_flat_dim: Optional[int] = None,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Ejecuta sanity checks para un bundle con estructura:

    {
      "target": "t2_dir_thr_90",
      "horizon": 90,
      "train": {"X":..., "y":...},
      "valid": {"X":..., "y":...},
      "test":  {"X":..., "y":...},
    }

    Detecta automáticamente si X es 2D o 3D, y usa TRAIN como referencia
    si no se pasan expected_*.
    """
    X_tr = bundle["train"]["X"]
    y_tr = bundle["train"]["y"]
    X_va = bundle["valid"]["X"]
    y_va = bundle["valid"]["y"]
    X_te = bundle["test"]["X"]
    y_te = bundle["test"]["y"]

    X_tr_arr = np.asarray(X_tr)
    seq_len_tr, n_feat_tr, mode_tr = _infer_seq_len_and_n_features(X_tr_arr)

    # --------------------------------------------------
    # Setear esperados desde TRAIN si no se dieron
    # --------------------------------------------------
    if mode_tr == "3d":
        if expected_seq_len is None:
            expected_seq_len = seq_len_tr
        if expected_n_features is None:
            expected_n_features = n_feat_tr
        expected_flat_dim = None
    else:
        d_flat_tr = int(X_tr_arr.shape[1])
        if expected_flat_dim is None:
            expected_flat_dim = d_flat_tr
        expected_n_features = None

    # --------------------------------------------------
    # Ejecutar checks
    # --------------------------------------------------
    out_tr = sanity_check_seq2one(
        X_tr,
        y_tr,
        f"train_{tag}",
        expected_seq_len=expected_seq_len,
        expected_n_features=expected_n_features,
        expected_flat_dim=expected_flat_dim,
        verbose=verbose,
    )

    out_va = sanity_check_seq2one(
        X_va,
        y_va,
        f"valid_{tag}",
        expected_seq_len=expected_seq_len,
        expected_n_features=expected_n_features,
        expected_flat_dim=expected_flat_dim,
        verbose=verbose,
    )

    out_te = sanity_check_seq2one(
        X_te,
        y_te,
        f"test_{tag}",
        expected_seq_len=expected_seq_len,
        expected_n_features=expected_n_features,
        expected_flat_dim=expected_flat_dim,
        verbose=verbose,
    )

    target = bundle.get("target", "NA")
    horizon = bundle.get("horizon", "NA")

    if verbose:
        print(f"OK {tag} | target={target} | horizon={horizon}")

    return {
        "target": target,
        "horizon": horizon,
        "train": out_tr,
        "valid": out_va,
        "test": out_te,
    }


def run_sanity_checks_all_targets_seq2one(
    bundle_t2_90: Dict[str, Any],
    bundle_t2_120: Dict[str, Any],
    *,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Corre sanity checks para ambos targets T2.
    """
    out_t2_90 = run_sanity_checks_for_bundle_seq2one(
        bundle_t2_90,
        tag="t2_90",
        verbose=verbose,
    )

    out_t2_120 = run_sanity_checks_for_bundle_seq2one(
        bundle_t2_120,
        tag="t2_120",
        verbose=verbose,
    )

    return {
        "t2_dir_thr_90": out_t2_90,
        "t2_dir_thr_120": out_t2_120,
    }

In [14]:
#sanity_outputs = run_sanity_checks_all_targets_seq2one(
#    bundle_t2_90,
#    bundle_t2_120,
#    verbose=True,
#)

In [15]:
#summary_delta = run_sanity_checks_all_horizons_seq2one(bundle_delta_60, bundle_delta_90)
#summary_ret = run_sanity_checks_all_horizons_seq2one(bundle_ret_60, bundle_ret_90)

## **6. Métricas de clasificación T2**

In [16]:
# ================================
# Setup para importar módulos del proyecto
# ================================

if str(DRIVE_DIR) not in sys.path:
    sys.path.insert(0, str(DRIVE_DIR))

(DRIVE_DIR / "metrics" / "__init__.py").touch(exist_ok=True)

importlib.invalidate_caches()

# ================================
# Imports de métricas (T2 clasificación)
# ================================

from metrics.classification_metrics import (
    compute_classification_metrics,
    metrics_to_flat_dict,
    print_classification_report_block,
)

# ================================
# Utilidad: métricas → DataFrame
# ================================

import pandas as pd

def metrics_to_df(
    metrics: dict,
    *,
    model: str,
    split: str,
    window_size: int,
    target: str,
) -> pd.DataFrame:
    """
    Convierte métricas de clasificación T2 en una fila de DataFrame.
    """

    m = metrics["metrics"]

    return pd.DataFrame([{
        "model": model,
        "split": split,
        "window_size": window_size,
        "target": target,
        "n_samples": m["n_samples"],

        # principales
        "balanced_accuracy": m["balanced_accuracy"],
        "f1_macro": m["f1_macro"],
        "f1_weighted": m["f1_weighted"],

        # complementarias
        "accuracy": m["accuracy"],
        "precision_macro": m["precision_macro"],
        "recall_macro": m["recall_macro"],

        # baseline
        "balanced_accuracy_naive": m.get("balanced_accuracy_naive"),
        "bal_acc_gain_vs_naive": m.get("balanced_accuracy_gain_vs_naive"),
    }])

logger.info("Métricas T2 + utilidades cargadas")

2026-04-08 19:47:27,415 | INFO | Métricas T2 + utilidades cargadas


## **7. Persistencia de métricas (tracking de experimentos)**

In [17]:
# ================================
# Carga de métricas (si existen)
# ================================

def load_classification_metrics_if_exists(
    *,
    name: str,
    base_dir: Path = DRIVE_DIR / "metrics/classification_metrics",
) -> pd.DataFrame:
    """
    Carga métricas de clasificación si el archivo existe.
    """

    path = base_dir / f"classification_{name}_metrics.parquet"

    if path.exists():
        logger.info(f"Cargando métricas desde: {path}")
        return pd.read_parquet(path)

    logger.info(f"No existen métricas previas para: {name}")
    return pd.DataFrame()

In [18]:
# ================================
# Guardado de métricas
# ================================

def save_classification_metrics(
    df_metrics: pd.DataFrame,
    *,
    name: str,
    base_dir: Path = DRIVE_DIR / "metrics/classification_metrics",
) -> Path:
    """
    Guarda un DataFrame de métricas de clasificación en formato Parquet.
    """

    base_dir.mkdir(parents=True, exist_ok=True)

    out_path = base_dir / f"classification_{name}_metrics.parquet"

    df_metrics.to_parquet(out_path, index=False)

    logger.info(f"Métricas guardadas en: {out_path}")

    return out_path

Ejemplo de uso:

```python
df_metrics = load_classification_metrics_if_exists(name="lstm_valid")

df_metrics = pd.concat([df_metrics, new_row_df], ignore_index=True)

save_classification_metrics(df_metrics, name="lstm_valid")
```



## **8. Gestión de dispositivo y memoria**

In [19]:
# ================================
# Device y limpieza de memoria
# ================================

def get_torch_device():
    """
    Retorna el device para modelos PyTorch.
    """
    import torch

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    logger.info(f"Using device: {device}")
    return device


def clear_torch_memory() -> None:
    """
    Limpia memoria Python y, si existe CUDA, libera caché GPU.
    Útil entre entrenamientos de modelos PyTorch.
    """
    import gc

    gc.collect()

    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except ImportError:
        pass

## **9. Reproducibilidad**

In [20]:
# ================================
# Reproducibilidad
# ================================

def set_seeds(seed: int = 42) -> None:
    """
    Fija semillas para reproducibilidad en:
    - Python
    - NumPy
    - PyTorch (si está disponible)
    """

    import random
    import numpy as np
    import os

    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

    # ---- PyTorch (si está instalado)
    try:
        import torch

        torch.manual_seed(seed)
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

        # determinismo (más lento pero reproducible)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

    except ImportError:
        pass  # no hay PyTorch, ignorar

    logger.info(f"Seeds fijadas en {seed}")


# Aplicación
SEED = 42
set_seeds(SEED)

2026-04-08 19:47:30,978 | INFO | Seeds fijadas en 42


# **10. Entrenamiento de modelo**

## **10.1. Función unitaria por bundle**

In [21]:
import copy
import numpy as np
import torch
import torch.nn as nn


class LSTMClassifier(nn.Module):
    def __init__(
        self,
        n_features: int,
        hidden_size: int = 64,
        num_layers: int = 1,
        dropout: float = 0.0,
        num_classes: int = 3,
    ):
        super().__init__()

        lstm_dropout = dropout if num_layers > 1 else 0.0

        self.lstm = nn.LSTM(
            input_size=n_features,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=lstm_dropout,
        )

        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        # x: (batch, seq_len, n_features)
        out, _ = self.lstm(x)         # (batch, seq_len, hidden_size)
        last_out = out[:, -1, :]      # many-to-one
        last_out = self.dropout(last_out)
        logits = self.fc(last_out)    # (batch, num_classes)
        return logits


def run_lstm_for_bundle_seq2one(
    bundle,
    *,
    hidden_size: int = 64,
    num_layers: int = 1,
    dropout: float = 0.0,
    learning_rate: float = 1e-3,
    weight_decay: float = 0.0,
    batch_size: int = 256,
    epochs: int = 20,
    patience: int = 5,
    random_state: int = 42,
    device: str | None = None,
    deterministic: bool = True,
    class_weight=None,
    verbose: bool = False,
):
    """
    Ejecuta LSTM para un bundle seq2one.

    - Usa TRAIN para fit
    - Usa VALID para early stopping
    - Predice en VALID y TEST
    - Devuelve predicciones
    - Soporta labels arbitrarias (ej. [-1, 0, 1]) mediante codificación interna

    Requisitos
    ----------
    X debe tener shape:
        (n_samples, seq_len, n_features)

    Notas
    -----
    - Si device=None, usa 'cuda' si está disponible; si no, 'cpu'
    - Si deterministic=True, fuerza mayor reproducibilidad

    class_weight : None | "balanced" | dict
        - None       -> entrenamiento natural
        - "balanced" -> ponderación automática por frecuencia inversa
        - dict       -> pesos manuales por clase original, ej. {-1: 2.0, 0: 1.0, 1: 2.0}
    """

    # =========================
    # 1. SEEDS Y DEVICE
    # =========================
    torch.manual_seed(random_state)
    np.random.seed(random_state)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(random_state)

    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    # =========================
    # 2. EXTRAER DATA
    # =========================
    X_train = bundle["train"]["X"]
    y_train = bundle["train"]["y"]

    X_valid = bundle["valid"]["X"]
    y_valid = bundle["valid"]["y"]

    X_test = bundle["test"]["X"]

    # =========================
    # 3. VALIDAR SHAPES
    # =========================
    if X_train.ndim != 3 or X_valid.ndim != 3 or X_test.ndim != 3:
        raise ValueError(
            "LSTM requiere tensores 3D: (n_samples, seq_len, n_features). "
            f"Recibido train={X_train.shape}, valid={X_valid.shape}, test={X_test.shape}"
        )

    seq_len_train, n_features_train = X_train.shape[1], X_train.shape[2]
    seq_len_valid, n_features_valid = X_valid.shape[1], X_valid.shape[2]
    seq_len_test,  n_features_test  = X_test.shape[1],  X_test.shape[2]

    if not (
        seq_len_train == seq_len_valid == seq_len_test
        and n_features_train == n_features_valid == n_features_test
    ):
        raise ValueError(
            "Inconsistencia entre shapes de train/valid/test. "
            f"train={X_train.shape}, valid={X_valid.shape}, test={X_test.shape}"
        )

    n_features = n_features_train

    # =========================
    # 4. CODIFICAR LABELS
    # =========================
    classes_ = np.sort(np.unique(y_train))

    unknown_valid = set(np.unique(y_valid)) - set(classes_)
    if unknown_valid:
        raise ValueError(
            f"VALID contiene clases no vistas en TRAIN: {sorted(unknown_valid)}"
        )

    class_to_idx = {cls: idx for idx, cls in enumerate(classes_)}
    idx_to_class = {idx: cls for cls, idx in class_to_idx.items()}

    y_train_enc = np.array([class_to_idx[y] for y in y_train], dtype=np.int64)
    y_valid_enc = np.array([class_to_idx[y] for y in y_valid], dtype=np.int64)

    num_classes = len(classes_)

    # =========================
    # 5. CLASS WEIGHTS
    # =========================
    criterion_weight = None
    weights_by_idx = None

    if class_weight is None:
        criterion_weight = None

    elif class_weight == "balanced":
        counts = np.bincount(y_train_enc, minlength=num_classes)
        total = counts.sum()

        weights_by_idx = {
            idx: total / (num_classes * count)
            for idx, count in enumerate(counts)
        }

        criterion_weight = torch.tensor(
            [weights_by_idx[idx] for idx in range(num_classes)],
            dtype=torch.float32,
            device=device,
        )

    elif isinstance(class_weight, dict):
        weights_by_idx = {
            class_to_idx[cls]: weight
            for cls, weight in class_weight.items()
            if cls in class_to_idx
        }

        criterion_weight = torch.tensor(
            [weights_by_idx.get(idx, 1.0) for idx in range(num_classes)],
            dtype=torch.float32,
            device=device,
        )

    else:
        raise ValueError(
            "class_weight debe ser None, 'balanced' o dict"
        )

    # =========================
    # 6. TENSORES
    # =========================
    X_train_t = torch.tensor(X_train, dtype=torch.float32)
    y_train_t = torch.tensor(y_train_enc, dtype=torch.long)

    X_valid_t = torch.tensor(X_valid, dtype=torch.float32).to(device)
    y_valid_t = torch.tensor(y_valid_enc, dtype=torch.long).to(device)

    X_test_t = torch.tensor(X_test, dtype=torch.float32).to(device)

    train_ds = torch.utils.data.TensorDataset(X_train_t, y_train_t)
    train_loader = torch.utils.data.DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        drop_last=False,
        num_workers=2,
        pin_memory=True,
    )

    # =========================
    # 7. MODELO
    # =========================
    model = LSTMClassifier(
        n_features=n_features,
        hidden_size=hidden_size,
        num_layers=num_layers,
        dropout=dropout,
        num_classes=num_classes,
    ).to(device)

    criterion = nn.CrossEntropyLoss(weight=criterion_weight)
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay,
    )

    # =========================
    # 8. TRAIN + EARLY STOPPING
    # =========================
    best_state = None
    best_valid_loss = np.inf
    best_epoch = 0
    wait = 0
    history = []

    try:
        for epoch in range(1, epochs + 1):
            model.train()
            train_loss_sum = 0.0
            train_count = 0

            for xb, yb in train_loader:
                xb = xb.to(device)
                yb = yb.to(device)

                optimizer.zero_grad()
                logits = model(xb)
                loss = criterion(logits, yb)
                loss.backward()
                optimizer.step()

                batch_n = xb.size(0)
                train_loss_sum += loss.item() * batch_n
                train_count += batch_n

            train_loss = train_loss_sum / max(train_count, 1)

            model.eval()
            with torch.no_grad():
                valid_logits = model(X_valid_t)
                valid_loss = criterion(valid_logits, y_valid_t).item()

            history.append(
                {
                    "epoch": epoch,
                    "train_loss": train_loss,
                    "valid_loss": valid_loss,
                }
            )

            if verbose:
                print(
                    f"[Epoch {epoch:03d}] "
                    f"train_loss={train_loss:.6f} | "
                    f"valid_loss={valid_loss:.6f}"
                )

            if valid_loss < best_valid_loss:
                best_valid_loss = valid_loss
                best_epoch = epoch
                best_state = copy.deepcopy(model.state_dict())
                wait = 0
            else:
                wait += 1
                if wait >= patience:
                    if verbose:
                        print(
                            f"[EARLY STOP] epoch={epoch} | "
                            f"best_epoch={best_epoch} | "
                            f"best_valid_loss={best_valid_loss:.6f}"
                        )
                    break

        if best_state is not None:
            model.load_state_dict(best_state)

        # =========================
        # 9. PREDICT
        # =========================
        model.eval()
        with torch.no_grad():
            valid_logits = model(X_valid_t)
            test_logits  = model(X_test_t)

            y_pred_valid_enc = valid_logits.argmax(dim=1).cpu().numpy()
            y_pred_test_enc  = test_logits.argmax(dim=1).cpu().numpy()

            y_proba_valid = torch.softmax(valid_logits, dim=1).cpu().numpy()
            y_proba_test  = torch.softmax(test_logits, dim=1).cpu().numpy()

        y_pred_valid = np.array([idx_to_class[int(y)] for y in y_pred_valid_enc])
        y_pred_test  = np.array([idx_to_class[int(y)] for y in y_pred_test_enc])

        return {
            "model": model,
            "classes_": classes_,
            "class_to_idx": class_to_idx,
            "idx_to_class": idx_to_class,
            "class_weight": class_weight,
            "criterion_weight": (
                criterion_weight.detach().cpu().numpy()
                if criterion_weight is not None else None
            ),
            "weights_by_idx": weights_by_idx,
            "history": history,
            "best_valid_loss": float(best_valid_loss),
            "best_epoch": int(best_epoch),
            "y_pred_valid": y_pred_valid,
            "y_pred_test": y_pred_test,
            "y_proba_valid": y_proba_valid,
            "y_proba_test": y_proba_test,
            "device": device,
        }

    finally:
        if device == "cuda":
            torch.cuda.empty_cache()

## **10.2. Función de evaluación sobre uno o más bundles**

In [22]:
from typing import Any, Dict, List, Sequence, Union
import pandas as pd


def eval_lstm_bundles(
    bundles: Union[Dict[str, Any], Sequence[Dict[str, Any]]],
    *,
    split: str = "valid",
    model_name: str = "lstm",
    hidden_size: int = 64,
    num_layers: int = 1,
    dropout: float = 0.0,
    learning_rate: float = 1e-3,
    weight_decay: float = 0.0,
    batch_size: int = 256,
    epochs: int = 20,
    patience: int = 5,
    random_state: int = 42,
    device: str | None = None,
    class_weight=None,
    verbose: bool = False,
) -> pd.DataFrame:
    """
    Evalúa LSTM para uno o varios bundles seq2one y
    retorna un DataFrame consolidado.

    Incluye soporte para clases desbalanceadas o balanceadas.

    class_weight:
        - None       -> entrenamiento natural
        - "balanced" -> ponderación automática
        - dict       -> pesos manuales
    """

    # --------------------------------------------------
    # 1) Normalizar entrada a lista
    # --------------------------------------------------
    if isinstance(bundles, dict):
        bundles_list: List[Dict[str, Any]] = [bundles]
    else:
        bundles_list = list(bundles)

    # --------------------------------------------------
    # 2) Validar split
    # --------------------------------------------------
    if split not in ("valid", "test"):
        raise ValueError("split debe ser 'valid' o 'test'")

    rows = []

    # --------------------------------------------------
    # 3) Iterar por bundles
    # --------------------------------------------------
    for bundle in bundles_list:
        target = bundle.get("target", None)
        window_size = int(bundle["window_size"])
        horizon = int(bundle.get("horizon", -1))

        if verbose:
            print(
                f"  -> L{window_size} | "
                f"target={target} | "
                f"split={split} | "
                f"model={model_name} | "
                f"class_weight={class_weight}"
            )

        # ----------------------------------------------
        # 4) Entrenar + predecir
        # ----------------------------------------------
        preds = run_lstm_for_bundle_seq2one(
            bundle,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout,
            learning_rate=learning_rate,
            weight_decay=weight_decay,
            batch_size=batch_size,
            epochs=epochs,
            patience=patience,
            random_state=random_state,
            device=device,
            class_weight=class_weight,
            verbose=False,
        )

        # ----------------------------------------------
        # 5) Seleccionar y_true / y_pred del split
        # ----------------------------------------------
        y_true = bundle[split]["y"]
        y_pred_key = f"y_pred_{split}"

        if y_pred_key not in preds:
            raise KeyError(
                f"No existe '{y_pred_key}' en la salida de "
                f"run_lstm_for_bundle_seq2one"
            )

        y_pred = preds[y_pred_key]

        # ----------------------------------------------
        # 6) Métricas de clasificación
        # ----------------------------------------------
        metrics = compute_classification_metrics(
            y_true=y_true,
            y_pred=y_pred,
            model_name=model_name,
            split=split,
            target=target,
            labels=[-1, 0, 1],
        )

        # ----------------------------------------------
        # 7) A DataFrame
        # ----------------------------------------------
        df_row = metrics_to_df(
            metrics,
            model=model_name,
            split=split,
            window_size=window_size,
            target=target,
        )

        df_row["horizon_min"] = horizon
        df_row["class_weight_mode"] = class_weight

        rows.append(df_row)

    # --------------------------------------------------
    # 8) Consolidar salida
    # --------------------------------------------------
    return pd.concat(rows, ignore_index=True)

## **10.3. Función orquestadora por `window_size`**

In [23]:
import gc
import pandas as pd


def run_lstm(
    window_size: int,
    *,
    verbose: bool = True,
    model_name: str = "lstm",
    hidden_size: int = 64,
    num_layers: int = 1,
    dropout: float = 0.0,
    learning_rate: float = 1e-3,
    weight_decay: float = 0.0,
    batch_size: int = 256,
    epochs: int = 20,
    patience: int = 5,
    random_state: int = 42,
    device: str | None = None,
    class_weight=None,
) -> pd.DataFrame:
    """
    Ejecuta LSTM para una sola window_size
    sobre los targets T2:
      - t2_dir_thr_90
      - t2_dir_thr_120

    Retorna un DataFrame consolidado con métricas de VALID y TEST.

    Notas
    -----
    - Si device=None, el modelo intenta usar GPU ('cuda') si está disponible.
    - Si no hay GPU disponible, usa CPU automáticamente.

    class_weight : None | "balanced" | dict
    """

    size = int(window_size)

    bundle_t2_90 = bundle_t2_120 = None
    bundles_t2 = None
    df_out = None

    model_name_effective = (
        f"{model_name}_balanced" if class_weight == "balanced" else model_name
    )

    try:
        # --------------------------------------------------
        # 1) Encabezado
        # --------------------------------------------------
        if verbose:
            print("\n" + "=" * 80)
            print(f"LSTM | T2 SEQ2ONE | WINDOW_SIZE=L{size}")
            print("=" * 80)
            print(f"class_weight = {class_weight}")

        # --------------------------------------------------
        # 2) Construcción de bundles para esta ventana
        # --------------------------------------------------
        if verbose:
            print(f"\n[BUILD] L{size} | targets = ['t2_dir_thr_90', 't2_dir_thr_120']")

        bundle_t2_90, bundle_t2_120 = create_bundles(
            window_size=size,
            targets=["t2_dir_thr_90", "t2_dir_thr_120"],
            windows_paths=WINDOWS_PATHS,
            scaler_path=SCALER_T2_PATH,
        )

        bundles_t2 = [bundle_t2_90, bundle_t2_120]

        # --------------------------------------------------
        # 3) Evaluación por split
        # --------------------------------------------------
        dfs = []

        for split in ["valid", "test"]:
            if verbose:
                print(
                    f"\n[EVAL] L{size} | split={split} | "
                    f"model={model_name_effective} | class_weight={class_weight}"
                )

            df_split = eval_lstm_bundles(
                bundles_t2,
                split=split,
                model_name=model_name_effective,
                hidden_size=hidden_size,
                num_layers=num_layers,
                dropout=dropout,
                learning_rate=learning_rate,
                weight_decay=weight_decay,
                batch_size=batch_size,
                epochs=epochs,
                patience=patience,
                random_state=random_state,
                device=device,
                class_weight=class_weight,
                verbose=verbose,
            )
            dfs.append(df_split)

        # --------------------------------------------------
        # 4) Consolidación final
        # --------------------------------------------------
        df_out = (
            pd.concat(dfs, ignore_index=True)
            .sort_values(["window_size", "target", "split", "horizon_min", "model"])
            .reset_index(drop=True)
        )

        # --------------------------------------------------
        # 5) Resumen final
        # --------------------------------------------------
        if verbose:
            print(f"\n[DONE] L{size} | rows={len(df_out)}")
            print(
                df_out[
                    [
                        "window_size",
                        "split",
                        "target",
                        "model",
                        "horizon_min",
                        "class_weight_mode",
                        "balanced_accuracy",
                        "f1_macro",
                    ]
                ]
                .sort_values(
                    ["split", "target", "model", "horizon_min", "class_weight_mode"]
                )
                .to_string(index=False)
            )

        return df_out

    finally:
        # --------------------------------------------------
        # 6) Liberación de memoria
        # --------------------------------------------------
        del bundle_t2_90, bundle_t2_120, bundles_t2
        gc.collect()

## **10.4. Función incremental multi-ventana**

In [24]:
from pathlib import Path
import pandas as pd


def run_lstm_incremental(
    *,
    window_sizes: list[int],
    name: str = "lstm",
    verbose: bool = True,
    hidden_size: int = 64,
    num_layers: int = 1,
    dropout: float = 0.0,
    learning_rate: float = 1e-3,
    weight_decay: float = 0.0,
    batch_size: int = 256,
    epochs: int = 20,
    patience: int = 5,
    random_state: int = 42,
    device: str | None = None,
    class_weight=None,
) -> pd.DataFrame:
    """
    Ejecuta LSTM de forma incremental para múltiples window_sizes.

    - Carga histórico si existe
    - Hace SKIP si un window_size ya está completo
    - Corre run_lstm(window_size=L) para los faltantes
    - Agrega resultados nuevos al histórico
    - Guarda usando save_classification_metrics(df_hist, name=name)

    Notas
    -----
    - Si device=None, el modelo intenta usar GPU ('cuda') si está disponible.
    - Si no hay GPU disponible, usa CPU automáticamente.

    class_weight : None | "balanced" | dict
    """

    # nombre efectivo del experimento
    name_effective = f"{name}_balanced" if class_weight == "balanced" else name

    metrics_dir = DRIVE_DIR / "metrics/classification_metrics"
    metrics_path = metrics_dir / f"classification_{name_effective}_metrics.parquet"

    # --------------------------------------------------
    # 1) Cargar histórico si existe
    # --------------------------------------------------
    if metrics_path.exists():
        df_hist = pd.read_parquet(metrics_path)
    else:
        df_hist = pd.DataFrame()

    # --------------------------------------------------
    # 2) Definir completitud esperada por window_size
    # --------------------------------------------------
    expected_targets = {"t2_dir_thr_90", "t2_dir_thr_120"}
    expected_splits = {"valid", "test"}
    expected_models = {name_effective}

    # --------------------------------------------------
    # 3) Iterar por window_sizes
    # --------------------------------------------------
    for L in window_sizes:
        L = int(L)

        # ----------------------------------------------
        # Skip robusto por window_size
        # ----------------------------------------------
        if not df_hist.empty:
            dfL = df_hist[df_hist["window_size"] == L]

            done_targets = set(dfL["target"].unique()) if not dfL.empty else set()
            done_splits = set(dfL["split"].unique()) if not dfL.empty else set()
            done_models = set(dfL["model"].unique()) if not dfL.empty else set()

            if class_weight == "balanced":
                done_class_weight_mode = (
                    set(dfL["class_weight_mode"].dropna().unique())
                    if ("class_weight_mode" in dfL.columns and not dfL.empty)
                    else set()
                )

                is_complete = (
                    expected_targets.issubset(done_targets)
                    and expected_splits.issubset(done_splits)
                    and expected_models.issubset(done_models)
                    and {"balanced"}.issubset(done_class_weight_mode)
                )
            else:
                is_complete = (
                    expected_targets.issubset(done_targets)
                    and expected_splits.issubset(done_splits)
                    and expected_models.issubset(done_models)
                )

            if is_complete:
                if verbose:
                    print(f"[SKIP] {name_effective} L={L} ya existe completo en Drive")
                continue

        # ----------------------------------------------
        # Ejecutar LSTM para este window_size
        # ----------------------------------------------
        if verbose:
            print("\n" + "-" * 80)
            print(f"[RUN] {name_effective} | L={L} | class_weight={class_weight}")
            print("-" * 80)

        df_L = run_lstm(
            window_size=L,
            verbose=verbose,
            model_name=name,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout,
            learning_rate=learning_rate,
            weight_decay=weight_decay,
            batch_size=batch_size,
            epochs=epochs,
            patience=patience,
            random_state=random_state,
            device=device,
            class_weight=class_weight,
        )

        # Etiqueta de familia
        df_L["family"] = name_effective

        # ----------------------------------------------
        # Actualizar histórico
        # ----------------------------------------------
        if df_hist.empty:
            df_hist = df_L.copy()
        else:
            df_hist = pd.concat([df_hist, df_L], ignore_index=True)

        # ----------------------------------------------
        # Eliminar duplicados por seguridad
        # ----------------------------------------------
        subset_cols = [
            "window_size",
            "target",
            "split",
            "model",
            "horizon_min",
        ]
        if "class_weight_mode" in df_hist.columns:
            subset_cols.append("class_weight_mode")

        df_hist = (
            df_hist
            .drop_duplicates(subset=subset_cols, keep="last")
            .reset_index(drop=True)
        )

        # ----------------------------------------------
        # Guardar histórico actualizado
        # ----------------------------------------------
        save_classification_metrics(df_hist, name=name_effective)

    # --------------------------------------------------
    # 4) Retorno final ordenado
    # --------------------------------------------------
    sort_cols = ["window_size", "target", "split", "horizon_min", "model"]
    if "class_weight_mode" in df_hist.columns:
        sort_cols.append("class_weight_mode")

    return df_hist.sort_values(sort_cols).reset_index(drop=True)

## **10.5. Ejecución final del experimento**

In [25]:
df_lstm_none = run_lstm_incremental(
    window_sizes=[30, 60, 90, 120, 180],
    name="lstm",
    class_weight=None,
)




--------------------------------------------------------------------------------
[RUN] lstm | L=30 | class_weight=None
--------------------------------------------------------------------------------

LSTM | T2 SEQ2ONE | WINDOW_SIZE=L30
class_weight = None

[BUILD] L30 | targets = ['t2_dir_thr_90', 't2_dir_thr_120']


2026-04-08 19:47:32,380 | INFO | Loaded: windows_t2_dir_thr_90_train.npz
2026-04-08 19:47:32,380 | INFO | X shape: (436692, 30, 5) | y shape: (436692,)
2026-04-08 19:47:33,134 | INFO | Loaded: windows_t2_dir_thr_90_valid.npz
2026-04-08 19:47:33,135 | INFO | X shape: (93508, 30, 5) | y shape: (93508,)
2026-04-08 19:47:33,644 | INFO | Loaded: windows_t2_dir_thr_90_test.npz
2026-04-08 19:47:33,645 | INFO | X shape: (93990, 30, 5) | y shape: (93990,)
2026-04-08 19:47:33,963 | INFO | Scaler cargado: scaler_t2.pkl
2026-04-08 19:47:33,963 | INFO | Bundle cargado | target=t2_dir_thr_90 | window_size=30 | train=(436692, 30, 5) | valid=(93508, 30, 5) | test=(93990, 30, 5)
2026-04-08 19:47:34,890 | INFO | Loaded: windows_t2_dir_thr_120_train.npz
2026-04-08 19:47:34,891 | INFO | X shape: (436692, 30, 5) | y shape: (436692,)
2026-04-08 19:47:35,505 | INFO | Loaded: windows_t2_dir_thr_120_valid.npz
2026-04-08 19:47:35,505 | INFO | X shape: (93508, 30, 5) | y shape: (93508,)
2026-04-08 19:47:36,043 |


WINDOW_SIZE: 30

TARGET: t2_dir_thr_90
Train : (436692, 30, 5) (436692,)
Valid : (93508, 30, 5) (93508,)
Test  : (93990, 30, 5) (93990,)
Scaler: StandardScaler

TARGET: t2_dir_thr_120
Train : (436692, 30, 5) (436692,)
Valid : (93508, 30, 5) (93508,)
Test  : (93990, 30, 5) (93990,)
Scaler: StandardScaler

[EVAL] L30 | split=valid | model=lstm | class_weight=None
  -> L30 | target=t2_dir_thr_90 | split=valid | model=lstm | class_weight=None
  -> L30 | target=t2_dir_thr_120 | split=valid | model=lstm | class_weight=None

[EVAL] L30 | split=test | model=lstm | class_weight=None
  -> L30 | target=t2_dir_thr_90 | split=test | model=lstm | class_weight=None
  -> L30 | target=t2_dir_thr_120 | split=test | model=lstm | class_weight=None

[DONE] L30 | rows=4
 window_size split         target model  horizon_min class_weight_mode  balanced_accuracy  f1_macro
          30  test t2_dir_thr_120  lstm          120              None           0.342512  0.265170
          30  test  t2_dir_thr_90  lstm 

2026-04-08 19:48:48,243 | INFO | Métricas guardadas en: /content/drive/MyDrive/neural_profit/metrics/classification_metrics/classification_lstm_metrics.parquet



--------------------------------------------------------------------------------
[RUN] lstm | L=60 | class_weight=None
--------------------------------------------------------------------------------

LSTM | T2 SEQ2ONE | WINDOW_SIZE=L60
class_weight = None

[BUILD] L60 | targets = ['t2_dir_thr_90', 't2_dir_thr_120']


...

2026-04-08 19:51:42,226 | INFO | Métricas guardadas en: /content/drive/MyDrive/neural_profit/metrics/classification_metrics/classification_lstm_metrics.parquet



[DONE] L120 | rows=4
 window_size split         target model  horizon_min class_weight_mode  balanced_accuracy  f1_macro
         120  test t2_dir_thr_120  lstm          120              None           0.343450  0.252782
         120  test  t2_dir_thr_90  lstm           90              None           0.346459  0.263430
         120 valid t2_dir_thr_120  lstm          120              None           0.333872  0.273275
         120 valid  t2_dir_thr_90  lstm           90              None           0.339438  0.283045

--------------------------------------------------------------------------------
[RUN] lstm | L=180 | class_weight=None
--------------------------------------------------------------------------------

LSTM | T2 SEQ2ONE | WINDOW_SIZE=L180
class_weight = None

[BUILD] L180 | targets = ['t2_dir_thr_90', 't2_dir_thr_120']


2026-04-08 19:51:44,343 | INFO | Loaded: windows_t2_dir_thr_90_train.npz
2026-04-08 19:51:44,344 | INFO | X shape: (300792, 180, 5) | y shape: (300792,)
2026-04-08 19:51:45,065 | INFO | Loaded: windows_t2_dir_thr_90_valid.npz
2026-04-08 19:51:45,065 | INFO | X shape: (64408, 180, 5) | y shape: (64408,)
2026-04-08 19:51:45,920 | INFO | Loaded: windows_t2_dir_thr_90_test.npz
2026-04-08 19:51:45,921 | INFO | X shape: (64740, 180, 5) | y shape: (64740,)
2026-04-08 19:51:45,922 | INFO | Scaler cargado: scaler_t2.pkl
2026-04-08 19:51:45,922 | INFO | Bundle cargado | target=t2_dir_thr_90 | window_size=180 | train=(300792, 180, 5) | valid=(64408, 180, 5) | test=(64740, 180, 5)
2026-04-08 19:51:47,503 | INFO | Loaded: windows_t2_dir_thr_120_train.npz
2026-04-08 19:51:47,504 | INFO | X shape: (300792, 180, 5) | y shape: (300792,)
2026-04-08 19:51:48,261 | INFO | Loaded: windows_t2_dir_thr_120_valid.npz
2026-04-08 19:51:48,262 | INFO | X shape: (64408, 180, 5) | y shape: (64408,)
2026-04-08 19:51


WINDOW_SIZE: 180

TARGET: t2_dir_thr_90
Train : (300792, 180, 5) (300792,)
Valid : (64408, 180, 5) (64408,)
Test  : (64740, 180, 5) (64740,)
Scaler: StandardScaler

TARGET: t2_dir_thr_120
Train : (300792, 180, 5) (300792,)
Valid : (64408, 180, 5) (64408,)
Test  : (64740, 180, 5) (64740,)
Scaler: StandardScaler

[EVAL] L180 | split=valid | model=lstm | class_weight=None
  -> L180 | target=t2_dir_thr_90 | split=valid | model=lstm | class_weight=None


OutOfMemoryError: CUDA out of memory. Tried to allocate 58.70 GiB. GPU 0 has a total capacity of 94.97 GiB of which 34.93 GiB is free. Including non-PyTorch memory, this process has 60.04 GiB memory in use. Of the allocated memory 3.51 GiB is allocated by PyTorch, and 55.87 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
df_lstm_balanced = run_lstm_incremental(
    window_sizes=[30, 60, 90, 120, 180],
    name="lstm",
    class_weight="balanced",
)